## Graded Assignment 3

Integrating Feast Feature Store into the IRIS Pipeline

In this assignment, we need to add a feature store layer to our IRIS pipeline so that training and inference share a single, consistent source of engineered features - eliminating a common source of training/serving skew.

In [1]:
# For removing Deprecation Warning related to pyparsing
!pip install --upgrade matplotlib pyparsing

In [2]:
import warnings
warnings.filterwarnings("ignore")

## Step 1: Install feast, scikit-learn

Install feast, gcp dependencies and scikit-learn

In [3]:
!pip install feast scikit-learn feast[gcp]

In [4]:
!feast version

Feast SDK Version: "0.60.0"


In [5]:
!git init

hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>
Initialized empty Git repository in /home/jupyter/main_run/.git/


In [6]:
%%writefile .gitignore

# Ignore Feast local database files
feast_iris/feature_repo/data/*.db

# Ignore Jupyter checkpoints
.ipynb_checkpoints/

# Ignore the parquet file
feast_iris/feature_repo/data/iris_dataset.parquet

Writing .gitignore


In [7]:
!feast init --minimal feast_iris


Creating a new Feast repository in /home/jupyter/main_run/feast_iris.



## Step 2: Prepare the iris data

In [8]:
!gsutil cp gs://mlops-assignment-mlops-iitmadras-mainrun/data/raw/iris.csv iris.csv

Copying gs://mlops-assignment-mlops-iitmadras-mainrun/data/raw/iris.csv...
/ [1 files][  3.8 KiB/  3.8 KiB]                                                
Operation completed over 1 objects/3.8 KiB.                                      


In [9]:
!mkdir feast_iris/feature_repo/data

In [11]:
import pandas as pd

df = pd.read_csv("iris.csv") 

# Create a unique Entity ID
df['sample_id'] = range(1, len(df) + 1)
df['sample_id'] = df['sample_id'].astype('int64')

# Create an Event Timestamp (since static sp using timestamp of 1 month prior)
timestamp = pd.Timestamp.now(tz='UTC') - pd.Timedelta(days=30)
df['event_timestamp'] = timestamp

print(df)

# Save as Parquet
df.to_parquet("feast_iris/feature_repo/data/iris_dataset.parquet", index=False)
print("Data prepared and saved to iris_dataset.parquet")

     sepal_length  sepal_width  petal_length  petal_width    species  \
0             5.1          3.5           1.4          0.2     setosa   
1             4.9          3.0           1.4          0.2     setosa   
2             4.7          3.2           1.3          0.2     setosa   
3             4.6          3.1           1.5          0.2     setosa   
4             5.0          3.6           1.4          0.2     setosa   
..            ...          ...           ...          ...        ...   
145           6.7          3.0           5.2          2.3  virginica   
146           6.3          2.5           5.0          1.9  virginica   
147           6.5          3.0           5.2          2.0  virginica   
148           6.2          3.4           5.4          2.3  virginica   
149           5.9          3.0           5.1          1.8  virginica   

     sample_id                  event_timestamp  
0            1 2026-01-30 10:07:17.245830+00:00  
1            2 2026-01-30 10:07:17.

## Step 3: Configure Feature Store

In [12]:
%%writefile feast_iris/feature_repo/feature_store.yaml
project: feast_iris
registry: data/registry.db
provider: local
online_store:
    type: sqlite
    path: data/online_store.db
entity_key_serialization_version: 3

Overwriting feast_iris/feature_repo/feature_store.yaml


## Step 4: Define Entities, Data Sources & Feature Views

In [13]:
%%writefile feast_iris/feature_repo/iris_features.py
from datetime import timedelta
from feast import Entity, FeatureView, Field, FileSource, ValueType
from feast.types import Float32, String

# Define the Data Source
iris_source = FileSource(
    path="data/iris_dataset.parquet", 
    timestamp_field="event_timestamp"
)

# Define the Entity
iris_entity = Entity(
    name="sample_id", 
    value_type=ValueType.INT64, 
    description="Unique identifier for an Iris flower sample"
)

# Define the Feature View
iris_feature_view = FeatureView(
    name="iris_features",
    entities=[iris_entity],
    ttl=timedelta(weeks=52),
    schema=[
        Field(name="sepal_length", dtype=Float32),
        Field(name="sepal_width", dtype=Float32),
        Field(name="petal_length", dtype=Float32),
        Field(name="petal_width", dtype=Float32),
        Field(name="species", dtype=String), 
    ],
    online=True,
    source=iris_source,
)

Writing feast_iris/feature_repo/iris_features.py


## Step 5: Apply Definitions & Materialize Features

In [14]:
%cd feast_iris/feature_repo
!feast apply

/home/jupyter/main_run/feast_iris/feature_repo
No project found in the repository. Using project name feast_iris defined in feature_store.yaml
Applying changes for project feast_iris
Created project feast_iris
Created entity sample_id
Created feature view iris_features

Created sqlite table feast_iris_iris_features



In [15]:
!ls data -la

total 36
drwxr-xr-x 2 jupyter jupyter  4096 Mar  1 10:09 .
drwxr-xr-x 5 jupyter jupyter  4096 Mar  1 10:09 ..
-rw-r--r-- 1 jupyter jupyter  6649 Mar  1 10:07 iris_dataset.parquet
-rw-r--r-- 1 jupyter jupyter 16384 Mar  1 10:09 online_store.db
-rw-r--r-- 1 jupyter jupyter   951 Mar  1 10:09 registry.db


In [16]:
# Materialize features in the online store upto current time
!feast materialize-incremental $(date -u +"%Y-%m-%dT%H:%M:%S")

Materializing 1 feature views to 2026-03-01 10:10:42+00:00 into the sqlite online store.

iris_features from 2025-03-02 10:10:49+00:00 to 2026-03-01 10:10:42+00:00:


In [17]:
!ls data -la

total 168
drwxr-xr-x 2 jupyter jupyter   4096 Mar  1 10:10 .
drwxr-xr-x 5 jupyter jupyter   4096 Mar  1 10:09 ..
-rw-r--r-- 1 jupyter jupyter   6649 Mar  1 10:07 iris_dataset.parquet
-rw-r--r-- 1 jupyter jupyter 151552 Mar  1 10:10 online_store.db
-rw-r--r-- 1 jupyter jupyter    975 Mar  1 10:10 registry.db


## Step 6: Fetch Features for Offline Training

In [18]:
%cd ..

import pandas as pd
from feast import FeatureStore

store = FeatureStore(repo_path="feature_repo")

# Create the entity dataframe
entity_df = pd.read_parquet("feature_repo/data/iris_dataset.parquet")[['sample_id', 'event_timestamp']]

# Fetch historical features
training_data = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "iris_features:sepal_length",
        "iris_features:sepal_width",
        "iris_features:petal_length",
        "iris_features:petal_width",
        "iris_features:species"
    ]
).to_df()

training_data.head()

/home/jupyter/main_run/feast_iris


,sample_id,event_timestamp,sepal_length,sepal_width,petal_length,petal_width,species
0,1,2026-01-30 10:07:17.245830+00:00,5.1,3.5,1.4,0.2,setosa
1,97,2026-01-30 10:07:17.245830+00:00,5.7,2.9,4.2,1.3,versicolor
2,98,2026-01-30 10:07:17.245830+00:00,6.2,2.9,4.3,1.3,versicolor
3,99,2026-01-30 10:07:17.245830+00:00,5.1,2.5,3.0,1.1,versicolor
4,100,2026-01-30 10:07:17.245830+00:00,5.7,2.8,4.1,1.3,versicolor


In [19]:
%cd ..

import joblib
from sklearn.tree import DecisionTreeClassifier

print("Training the model...")
X_train = training_data[['sepal_length','sepal_width','petal_length','petal_width']]
y_train = training_data.species
mod_dt = DecisionTreeClassifier(max_depth = 3, random_state = 1)
mod_dt.fit(X_train,y_train)
joblib.dump(mod_dt, "model.joblib")



/home/jupyter/main_run
Training the model...


['model.joblib']

## Step 7: Fetch Features for Inference

In [20]:
%cd feast_iris

inference_request = [{"sample_id": 5}]

# Fetch features from the online store
online_features = store.get_online_features(
    features=[
        "iris_features:sepal_length",
        "iris_features:sepal_width",
        "iris_features:petal_length",
        "iris_features:petal_width",
    ],
    entity_rows=inference_request
).to_dict()

inference_df = pd.DataFrame(online_features)

inference_df = inference_df[['sepal_length', 'sepal_width', 'petal_length', 'petal_width']]

inference_df

/home/jupyter/main_run/feast_iris


,sepal_length,sepal_width,petal_length,petal_width
0,5.0,3.6,1.4,0.2


## Step 8: Doing Inference

In [22]:
%cd ..
model = joblib.load("model.joblib")
prediction = model.predict(inference_df)
print(f"Predicted Species: {prediction}")

/home/jupyter/main_run
Predicted Species: ['setosa']


In [ ]:
!gsutil cp -r Feast_Assignment.ipynb gs://ga-3-assignment-file-final-mlops-iitmadras